# Assignment #1: Multi-class Diabetes Risk Prediction
**Alexandria University — Faculty of Engineering**  
**CSE: Pattern Recognition**  

This notebook implements:
- **Section 2**: Dataset preparation (split, preprocessing, feature selection, imbalance handling) → `data.py`
- **Section 3**: Custom Feedforward Neural Network (FNN) → `models.py`
- **Section 4**: Full evaluation (Accuracy, F1-Scores, Confusion Matrix) → `evaluate.py`
- **Section 5**: Analysis and comparison
- **Section 6 (Bonus)**: TensorBoard monitoring

> **Dataset:** `diabetes_012_health_indicators_BRFSS2015.csv` (Kaggle BRFSS 2015)  
> Place the CSV in the same directory as this notebook.

## 0. Imports & Setup

In [ ]:
import os, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from matplotlib.patches import Patch

import torch
warnings.filterwarnings('ignore')

# ── Fixed Random Seed ────────────────────────────────────────────────────────
from data import SEED
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## 1. Load Dataset

In [ ]:
from data import CSV_PATH, TARGET_COL

df = pd.read_csv(CSV_PATH)
print('Shape:', df.shape)
print('\nClass distribution:')
print(df[TARGET_COL].value_counts())
df.head()

## 2. Dataset Preparation

All data-prep logic lives in **`data.py`**.

### Pipeline summary
| Step | Detail |
|---|---|
| Split | Stratified 70 / 10 / 20 (train / val / test) |
| Scaling | `StandardScaler` fit **only on train** |
| Feature selection | `SelectKBest(mutual_info_classif, k=15)` fit **only on train** |
| Imbalance | SMOTE applied **only on train** (via `resample()`) |

In [ ]:
from data import prepare_data, resample, K_FEATURES

splits = prepare_data(use_feature_selection=True)

print("Shapes after preprocessing + feature selection:")
print("X_train:", splits.X_train.shape)
print("X_val:  ", splits.X_val.shape)
print("X_test: ", splits.X_test.shape)
print()
print(f"Selected ({K_FEATURES}):", splits.feature_names)

### 2.1 Train / Validation / Test split sizes

In [ ]:
print(f'Train : {splits.X_train.shape[0]:>7}  |  {dict(pd.Series(splits.y_train).value_counts().sort_index())}')
print(f'Val   : {splits.X_val.shape[0]:>7}  |  {dict(pd.Series(splits.y_val).value_counts().sort_index())}')
print(f'Test  : {splits.X_test.shape[0]:>7}  |  {dict(pd.Series(splits.y_test).value_counts().sort_index())}')

### 2.2 Class imbalance — SMOTE

In [ ]:
X_train_bal, y_train_bal = resample(splits.X_train, splits.y_train, method='smote')

print('Before SMOTE:', dict(pd.Series(splits.y_train).value_counts().sort_index()))
print('After  SMOTE:', dict(pd.Series(y_train_bal).value_counts().sort_index()))

# Distribution plot
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
CLASS_LABELS = ['No Diabetes (0)', 'Prediabetes (1)', 'Diabetes (2)']

for ax, (counts, title) in zip(axes, [
    (pd.Series(splits.y_train).value_counts().sort_index(), 'Before SMOTE'),
    (pd.Series(y_train_bal).value_counts().sort_index(),    'After SMOTE'),
]):
    ax.bar(CLASS_LABELS, counts, color=['#4CAF50', '#FF9800', '#F44336'])
    ax.set_title(title)
    ax.set_ylabel('Sample Count')
    ax.set_ylim(0, max(counts)*1.15)
    for i, v in enumerate(counts):
        ax.text(i, v + max(counts)*0.02, str(v), ha='center', fontsize=9)

plt.suptitle('Class Distribution — Training Set', fontweight='bold')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150)
plt.show()
print('Saved: class_distribution.png')

## 3. Custom Feedforward Neural Network (FNN)

### 3.1 Architecture

Implemented in **`models.py`** as `DiabetesFNN`.  
**Design choices:**
- Input → `BatchNorm` → 6 hidden blocks
- Each block: `Linear → BatchNorm → ReLU/Tanh → Dropout`
- Decreasing widths: 1024 → 512 → 256 → 128 → 64 → 32
- Output: 3 neurons (logits; softmax handled by `CrossEntropyLoss`)
- Optimizer: Adam + weight decay
- Loss: `CrossEntropyLoss` (with class weights for baseline) + `SoftF1Loss` (auxiliary)

In [ ]:
from models import DiabetesFNN

_dummy = DiabetesFNN(n_features=K_FEATURES)
print(_dummy)
total_params = sum(p.numel() for p in _dummy.parameters() if p.requires_grad)
print(f'\nTrainable parameters: {total_params:,}')

### 3.2 Train Baseline + Balanced FNN

- **Baseline**: trained on imbalanced data with class-weighted CE + SoftF1
- **Balanced**: trained on SMOTE-resampled data with plain CE + SoftF1

In [ ]:
from models import train_fnn

EPOCHS     = 50
BATCH_SIZE = 512
LR         = 1e-3

fnn_baseline, fnn_balanced = train_fnn(
    X_train=splits.X_train,
    y_train=splits.y_train,
    X_val=splits.X_val,
    y_val=splits.y_val,
    resample_method='smote',
    input_dim=splits.X_train.shape[1],
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    weight_decay=1e-4,
    tensorboard_logdir='runs',  # Bonus Section 6
)

## 4. Evaluation

Evaluation utilities are in **`evaluate.py`**.

### 4.1 Compute Metrics on Test Set

In [ ]:
from models   import predict_fnn
from evaluate import compute_metrics

preds_baseline = predict_fnn(fnn_baseline, splits.X_test)
preds_balanced = predict_fnn(fnn_balanced, splits.X_test)

results = []
results.append(compute_metrics(splits.y_test, preds_baseline, 'FNN Baseline (Imbalanced)'))
results.append(compute_metrics(splits.y_test, preds_balanced, 'FNN Balanced (SMOTE)'))

### 4.2 Confusion Matrices

In [ ]:
from evaluate import plot_confusion_matrix

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_confusion_matrix(splits.y_test, preds_baseline, 'FNN Baseline (Imbalanced)', axes[0])
plot_confusion_matrix(splits.y_test, preds_balanced, 'FNN Balanced (SMOTE)',      axes[1])
plt.suptitle('Confusion Matrices — Test Set (Row-Normalized)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150)
plt.show()
print('Saved: confusion_matrices.png')

### 4.3 Summary Table

In [ ]:
results_df = pd.DataFrame(results).set_index('Model').round(4)
print('\n===== Summary Table =====')
display(results_df)
results_df.to_csv('results_summary.csv')
print('Saved: results_summary.csv')

## 5. Analysis and Comparison

### 5.1 Metric Justification

For this **highly imbalanced multi-class dataset**, the most informative metrics are:

| Metric | What it measures | Suitability for imbalanced data |
|---|---|---|
| **Accuracy** | Overall correct predictions | ❌ Misleading — dominated by majority class (No Diabetes) |
| **Micro-F1** | Aggregates TP/FP/FN globally | ⚠️ Equivalent to accuracy here — biased toward majority |
| **Macro-F1** | Unweighted mean F1 per class | ✅ Treats all classes equally — penalizes poor minority class performance |
| **Weighted-F1** | Support-weighted mean F1 | ⚠️ Better than accuracy but still favors majority class |

**Conclusion:** `Macro-F1` is the most meaningful metric for this task.

### 5.2 Model Comparison Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
metrics = ['Accuracy', 'F1_Micro', 'F1_Macro', 'F1_Weighted']
x       = np.arange(len(metrics))
width   = 0.32

for i, (model_name, row) in enumerate(results_df.iterrows()):
    bars = ax.bar(x + i*width, row[metrics], width, label=model_name, alpha=0.85)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.004,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)

ax.set_xticks(x + width/2)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('FNN Performance Comparison — Baseline vs SMOTE', fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150)
plt.show()
print('Saved: model_comparison.png')

## 6 (Bonus): TensorBoard Monitoring

Inside `train_fnn()` in `models.py`, `torch.utils.tensorboard.SummaryWriter` logs train/val loss and accuracy per epoch under `runs/FNN_Baseline` and `runs/FNN_SMOTE`.

To view the dashboard:
```bash
tensorboard --logdir=runs
```
Then open http://localhost:6006

In [ ]:
# Optional: launch TensorBoard inline
import os
from tensorboard import program

logdir = os.path.join(os.getcwd(), 'runs')
tb = program.TensorBoard()
tb.configure(argv=[None, '--logdir', logdir, '--port', '6006'])
url = tb.launch()
print(f'TensorBoard started at {url}')

## 7. Final Summary

All outputs generated by this notebook:

| File | Description |
|---|---|
| `class_distribution.png` | Class counts before/after SMOTE |
| `confusion_matrices.png` | Row-normalized CMs for both models |
| `model_comparison.png` | Bar chart of all metrics |
| `results_summary.csv` | Numerical results table |
| `runs/` | TensorBoard logs |